# Method of Lines and Runge-Kutta

Connect spatial right-hand sides to Runge-Kutta time stages using NRPy
time-stepping utilities.

Navigation: [Index](../index.ipynb) |
Previous: [Wave Equation with NumPy](wave_equation_with_numpy.ipynb) |
Next: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)

## Learning Goals

- Separate space derivatives from time stepping.
- Read an RK4 table as a recipe for intermediate stages.
- See how NRPy records a time-step function for generated projects.

## Words for This Notebook

- **Method of Lines:** first approximate space derivatives, then solve the
  remaining time problem as ordinary differential equations.
- **MoL:** short for Method of Lines.
- **Stage:** one intermediate calculation inside a Runge-Kutta time step.
- **Butcher table:** the table of numbers that defines a Runge-Kutta method.
- **Right-hand side:** the formula that gives the time change of a field.
- **Stage storage:** temporary grid-field arrays used during intermediate
  Runge-Kutta calculations.
- **Function prototype:** a C function declaration that lists the name,
  return type, and arguments.

Use the code cells actively: first predict what should happen, then run the
cell, then explain the output in plain language. This predict-run-explain
pattern keeps the physics idea connected to the programming details.

## Notation: RK4 Stages and NRPy Function Storage

Think of all evolved grid fields at one time as one vector \(U_n\). A
right-hand-side routine evaluates \(R(U)\), the time derivative of that
vector. RK4 asks for four right-hand-side evaluations, combines them with
Butcher-table weights, and writes the next field vector \(U_{n+1}\).

NRPy turns that algorithm into generated C using named storage arrays:

| Name | Role in the RK4 update |
| --- | --- |
| `y_n_gfs` | Current evolved grid fields, \(U_n\). |
| `k_odd_gfs`, `k_even_gfs` | Alternating storage for intermediate stage data. |
| `y_nplus1_running_total_gfs` | Accumulator for the weighted RK4 sum. |
| `MoL_step_forward_in_time` | Generated driver that calls the RHS and RK stage kernels. |

The next sections connect the RK4 table to these storage names and then to
generated C calls.

## Setup: Import Time-Stepping Tools

These imports expose the NRPy time-stepping and infrastructure writers used
below.

In [1]:
import sympy as sp
import nrpy.c_function as cfc
from nrpy.infrastructures import BHaH
from nrpy.infrastructures.BHaH.MoLtimestepping.MoL_step_forward_in_time import (
    register_CFunction_MoL_step_forward_in_time,
)
from nrpy.infrastructures.BHaH.MoLtimestepping.rk_butcher_table_dictionary import (
    generate_Butcher_tables,
    intermediate_stage_gf_names_list,
    is_diagonal_Butcher,
    validate,
)

## Step 1: Inspect the RK4 Butcher Table

The table rows list stage times and stage weights. The final row lists the
weights used to combine the four stages into one RK4 update.

Before running the cell, predict how many right-hand-side evaluations RK4
needs and which temporary storage names should appear.

In [2]:
butcher_tables = generate_Butcher_tables()
rk4_table, rk4_order = butcher_tables["RK4"]
rk4_stage_rows = rk4_table[:-1]
rk4_weight_row = rk4_table[-1]
rk4_storage_names = intermediate_stage_gf_names_list(butcher_tables, "RK4")
formatted_rows = []
for stage, row in enumerate(rk4_stage_rows, start=1):
    c_value = row[0]
    a_values = list(row[1:]) + [""] * (4 - len(row[1:]))
    formatted_rows.append((stage, c_value, *a_values[:4]))
print("stage c a1 a2 a3 a4")
for formatted_row in formatted_rows:
    print(*formatted_row)
print("b weights:", rk4_weight_row[1:])
print("RK4 order:", rk4_order)
print("is diagonal:", is_diagonal_Butcher(butcher_tables, "RK4"))
print("intermediate storage names:", rk4_storage_names)

stage c a1 a2 a3 a4
1 0    
2 1/2 1/2   
3 1/2 0 1/2  
4 1 0 0 1 
b weights: [1/6, 1/3, 1/3, 1/6]
RK4 order: 4
is diagonal: True
intermediate storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']


## Validation Check: Confirm RK4 Table and Storage Names

This check confirms that NRPy's RK4 entry has order 4, four stages, a final
weights row, and the expected intermediate storage names before code
generation.

In [3]:
validate(butcher_tables, ["RK4"], verbose=False)
expected_rk4_table = [
    [0],
    [sp.Rational(1, 2), sp.Rational(1, 2)],
    [sp.Rational(1, 2), 0, sp.Rational(1, 2)],
    [1, 0, 0, 1],
    [
        "",
        sp.Rational(1, 6),
        sp.Rational(1, 3),
        sp.Rational(1, 3),
        sp.Rational(1, 6),
    ],
]
expected_storage_names = [
    "y_nplus1_running_total_gfs",
    "k_odd_gfs",
    "k_even_gfs",
]
print("validated RK4 order:", rk4_order)
print("stage rows:", len(rk4_stage_rows))
print("storage names:", rk4_storage_names)
if rk4_order != 4:
    raise RuntimeError("Expected RK4 to report order 4.")
if rk4_table != expected_rk4_table:
    raise RuntimeError("Expected the classical RK4 Butcher coefficients.")
if rk4_storage_names != expected_storage_names:
    raise RuntimeError("Expected RK4 to use the standard NRPy stage storage.")
print("RK4 table and storage check passed")

RK4: PASSED VALIDATION
validated RK4 order: 4
stage rows: 4
storage names: ['y_nplus1_running_total_gfs', 'k_odd_gfs', 'k_even_gfs']
RK4 table and storage check passed


## Step 2: Map RK4 Rows to NRPy Stage Storage

The Butcher table gives the stage weights. NRPy also needs named arrays that
can hold the current fields, temporary stage data, and the running total. The
next cell prints that storage map in the order a generated project uses it.

In [4]:
stage_storage_rows = [
    ("current fields", "y_n_gfs", "input state U_n"),
    ("running total", "y_nplus1_running_total_gfs", "weighted sum for U_{n+1}"),
    ("odd stages", "k_odd_gfs", "stage 1 and stage 3 data"),
    ("even stages", "k_even_gfs", "stage 2 and stage 4 data"),
]
print("role | storage name | purpose")
for role, storage_name, purpose in stage_storage_rows:
    print(role, "|", storage_name, "|", purpose)

role | storage name | purpose
current fields | y_n_gfs | input state U_n
running total | y_nplus1_running_total_gfs | weighted sum for U_{n+1}
odd stages | k_odd_gfs | stage 1 and stage 3 data
even stages | k_even_gfs | stage 2 and stage 4 data


## Step 3: Register the RK Step-Forward Function

NRPy's stored function list holds a complete C function or workflow hook for
later file generation. After registration, each RK substep records which
storage it writes and which expression it uses.

In [5]:
cfc.CFunction_dict.pop("MoL_step_forward_in_time", None)
for name in ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]:
    BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict.pop(name, None)
register_CFunction_MoL_step_forward_in_time(
    butcher_tables,
    "RK4",
    rhs_string="rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);",
)
registered = cfc.CFunction_dict["MoL_step_forward_in_time"]
registered_substeps = BHaH.MoLtimestepping.rk_substep.MoL_Functions_dict
print(
    "registered MoL function names:",
    sorted(name for name in cfc.CFunction_dict if "MoL" in name or "mol" in name),
)
print("complete C function declaration:")
print(registered.function_prototype)
print("registered substep names:", sorted(registered_substeps))
print("registered RK stage actions:")
print("stage | launcher | writes")
for stage_name, rk_function in sorted(registered_substeps.items()):
    writes = ", ".join(rk_function.RK_lhs_str_list)
    print(stage_name, "|", rk_function.name, "|", writes)

registered MoL function names: ['MoL_step_forward_in_time']
complete C function declaration:
void MoL_step_forward_in_time(commondata_struct *restrict commondata, griddata_struct *restrict griddata);
registered substep names: ['RK_STEP1', 'RK_STEP2', 'RK_STEP3', 'RK_STEP4']
registered RK stage actions:
stage | launcher | writes
RK_STEP1 | rk_substep_1__launcher | y_nplus1_running_total_gfs[i], k_odd_gfs[i]
RK_STEP2 | rk_substep_2__launcher | y_nplus1_running_total_gfs[i], k_even_gfs[i]
RK_STEP3 | rk_substep_3__launcher | y_nplus1_running_total_gfs[i], k_odd_gfs[i]
RK_STEP4 | rk_substep_4__launcher | y_n_gfs[i]


## Step 4: Inspect the Generated RK Stage Schedule

The schedule lines show how the Method of Lines driver advances through all
four RK4 stages. On a first pass, look for:

- four `START k... substep` markers;
- the right-hand-side function call;
- one RK launcher call per stage;

The complete first substep block is printed after the schedule as one concrete
example of the generated stage kernel.

In [6]:
full_function = registered.full_function
schedule_lines = []
for line in full_function.splitlines():
    stripped = line.strip()
    if (
        stripped.startswith("// -={ START k")
        or stripped.startswith("rhs_eval(")
        or stripped.startswith("rk_substep_")
    ):
        schedule_lines.append(stripped)
print("generated RK stage schedule:")
for line in schedule_lines:
    print(line)
start = full_function.index("static void rk_substep_1_host")
end_marker = "} // END FUNCTION: rk_substep_1_host"
end = full_function.index(end_marker, start) + len(end_marker)
print("complete generated first RK substep block:")
print(full_function[start:end])

generated RK stage schedule:
rk_substep_1_host(params, k_odd_gfs, y_n_gfs, y_nplus1_running_total_gfs, dt);
rk_substep_2_host(params, k_even_gfs, y_nplus1_running_total_gfs, y_n_gfs, dt);
rk_substep_3_host(params, k_odd_gfs, y_nplus1_running_total_gfs, y_n_gfs, dt);
rk_substep_4_host(params, k_even_gfs, y_n_gfs, y_nplus1_running_total_gfs, dt);
// -={ START k1 substep }=-
rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
rk_substep_1__launcher(params, k_odd_gfs, y_n_gfs, y_nplus1_running_total_gfs, commondata->dt);
// -={ START k2 substep }=-
rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
rk_substep_2__launcher(params, k_even_gfs, y_nplus1_running_total_gfs, y_n_gfs, commondata->dt);
// -={ START k3 substep }=-
rhs_eval(commondata, params, rfmstruct, auxevol_gfs, in_gfs, rhs_gfs);
rk_substep_3__launcher(params, k_odd_gfs, y_nplus1_running_total_gfs, y_n_gfs, commondata->dt);
// -={ START k4 substep }=-
rhs_eval(commondata, params, rfmstruct,

## Validation Check: Confirm the Generated RK Stage Schedule

This check confirms that the registered RK4 function contains the four stage
markers and launcher calls expected from the Butcher table.

In [7]:
expected_substep_names = ["RK_STEP1", "RK_STEP2", "RK_STEP3", "RK_STEP4"]
if sorted(registered_substeps) != expected_substep_names:
    raise RuntimeError("Expected four registered RK4 substeps.")
for stage_number in range(1, 5):
    marker = f"START k{stage_number} substep"
    launcher = f"rk_substep_{stage_number}__launcher"
    if marker not in full_function:
        raise RuntimeError(f"Missing generated schedule marker: {marker}")
    if launcher not in full_function:
        raise RuntimeError(f"Missing generated launcher call: {launcher}")
print("generated RK4 stage schedule check passed")

generated RK4 stage schedule check passed


The RK4 validation confirms the stage table, and the generated schedule shows
how NRPy turns those stages into C. This is the time-integration driver used
by generated wave projects.

## Learning Check

Explain why a zero table/storage residual is not enough by itself: the
generated schedule must also call the RHS and one launcher for each RK4 stage.

## Continue to Boundary Conditions
- [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)